Module 04 - Model Preparation

Objective

The objective of this module is to transform the feature engineered dataset into a machine learning ready dataset.

The following tasks are performed:

- Dataset loading
- Feature type identification
- Train-Test Split
- Categorical Encoding
- Feature Scaling
- Feature Selection
- Final dataset generation

Care is taken to prevent data leakage by ensuring that all learnable transformations are fitted only on the training dataset.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    OrdinalEncoder,
    OneHotEncoder,
    StandardScaler
)

In [2]:
risk_df = pd.read_parquet(
    "../data/processed/credit_risk_feature_engineered.parquet"
)

print(risk_df.shape)

risk_df.head()

(1345310, 123)


,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,...,installment_income_ratio,fico_average,long_term_loan,high_revol_util,high_dti,prime_borrower,emp_years,interest_band,income_band,loan_band
0,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,leadman,10+ years,...,0.002237,677.0,0,0,0,0,10,Medium,Medium,Small
1,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,Engineer,10+ years,...,0.012620,717.0,0,0,0,0,10,Medium,Medium,Very Large
2,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,truck driver,10+ years,...,0.006868,697.0,1,0,0,0,10,Medium,Medium,Large
3,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,Contract Specialist,3 years,...,0.002776,697.0,1,0,0,0,3,Very High,Very High,Medium
4,11950.0,11950.0,11950.0,36 months,13.44,405.18,C,C3,Veterinary Tecnician,4 years,...,0.011917,692.0,0,0,0,0,4,Medium,Low,Medium


In [3]:
print(risk_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 1345310 entries, 0 to 1345309
Columns: 123 entries, loan_amnt to loan_band
dtypes: category(3), float64(91), int64(6), str(23)
memory usage: 1.4 GB
None


In [4]:
categorical_columns = risk_df.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

numeric_columns = risk_df.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Categorical Features :", len(categorical_columns))
print("Numerical Features :", len(numeric_columns))

Categorical Features : 26
Numerical Features : 97


In [5]:
categorical_summary = pd.DataFrame({
    "Feature": categorical_columns,
    "Unique Values": [
        risk_df[col].nunique()
        for col in categorical_columns
    ]
})

categorical_summary = categorical_summary.sort_values(
    by="Unique Values",
    ascending=False
)

categorical_summary

,Feature,Unique Values
3,emp_title,378354
10,desc,122004
12,title,61681
13,zip_code,943
15,earliest_cr_line,739
18,last_credit_pull_d,140
7,issue_d,139
17,last_pymnt_d,135
14,addr_state,51
2,sub_grade,35


In [6]:
categorical_summary.to_csv(
    "../reports/categorical_feature_summary.csv",
    index=False
)

print("Categorical feature summary saved.")

Categorical feature summary saved.


In [7]:
for column in categorical_columns:

    print("=" * 60)

    print(column)

    print("Unique Values :", risk_df[column].nunique())

    print(risk_df[column].value_counts().head(10))

term
Unique Values : 2
term
36 months    1020743
60 months     324567
Name: count, dtype: int64
grade
Unique Values : 7
grade
B    392741
C    381686
A    235090
D    200953
E     93650
F     32058
G      9132
Name: count, dtype: int64
sub_grade
Unique Values : 35
sub_grade
C1    85494
B4    83199
B5    82538
B3    81827
C2    79213
C3    74998
C4    74421
B2    74024
B1    71153
C5    67560
Name: count, dtype: int64
emp_title
Unique Values : 378354
emp_title
Unknown             85785
Teacher             21268
Manager             19470
Owner               10302
Registered Nurse     8774
RN                   8522
Supervisor           8289
Driver               7558
Sales                7487
Project Manager      6381
Name: count, dtype: int64
emp_length
Unique Values : 11
emp_length
10+ years    520710
2 years      121743
< 1 year     108061
3 years      107597
1 year        88494
5 years       84154
4 years       80556
6 years       62733
8 years       60701
7 years       59624
Name: cou

Feature Reduction

Before encoding, redundant, high-cardinality and leakage-prone categorical features are removed. This reduces dimensionality, improves model efficiency and prevents duplicate information from being learned.

In [8]:
columns_to_drop = [
    "term",
    "loan_status",
    "grade",
    "emp_length",
    "emp_title",
    "desc",
    "title",
    "zip_code",
    "issue_d",
    "last_credit_pull_d",
    "last_pymnt_d",
    "pymnt_plan",
    "hardship_flag"
]

risk_df.drop(
    columns=columns_to_drop,
    inplace=True
)

print("Remaining Columns :", risk_df.shape[1])

Remaining Columns : 110


Binary Encoding

Binary categorical variables are converted into numerical values to make them suitable for machine learning algorithms.

In [9]:
binary_mapping = {
    "Individual": 0,
    "Joint App": 1,
    "w": 0,
    "f": 1,
    "N": 0,
    "Y": 1,
    "Cash": 0,
    "DirectPay": 1
}

risk_df["application_type"] = risk_df["application_type"].map(binary_mapping)

risk_df["initial_list_status"] = risk_df["initial_list_status"].map(binary_mapping)

risk_df["debt_settlement_flag"] = risk_df["debt_settlement_flag"].map(binary_mapping)

risk_df["disbursement_method"] = risk_df["disbursement_method"].map(binary_mapping)

In [10]:
risk_df[
    [
        "application_type",
        "initial_list_status",
        "debt_settlement_flag",
        "disbursement_method"
    ]
].head()

,application_type,initial_list_status,debt_settlement_flag,disbursement_method
0,0,0,0,0
1,0,0,0,0
2,1,0,0,0
3,0,0,0,0
4,0,0,0,0


Ordinal Encoding

Ordered categorical variables are encoded while preserving their natural ordering.

In [11]:
grade_order = [
    "A1","A2","A3","A4","A5",
    "B1","B2","B3","B4","B5",
    "C1","C2","C3","C4","C5",
    "D1","D2","D3","D4","D5",
    "E1","E2","E3","E4","E5",
    "F1","F2","F3","F4","F5",
    "G1","G2","G3","G4","G5"
]

interest_order = [
    "Low",
    "Medium",
    "High",
    "Very High"
]

income_order = [
    "Low",
    "Medium",
    "High",
    "Very High"
]

loan_order = [
    "Small",
    "Medium",
    "Large",
    "Very Large"
]

In [12]:
ordinal_encoder = OrdinalEncoder(
    categories=[
        grade_order,
        interest_order,
        income_order,
        loan_order
    ]
)

risk_df[
    [
        "sub_grade",
        "interest_band",
        "income_band",
        "loan_band"
    ]
] = ordinal_encoder.fit_transform(
    risk_df[
        [
            "sub_grade",
            "interest_band",
            "income_band",
            "loan_band"
        ]
    ]
)

In [13]:
risk_df[
    [
        "sub_grade",
        "interest_band",
        "income_band",
        "loan_band"
    ]
].head()

,sub_grade,interest_band,income_band,loan_band
0,13.0,1.0,1.0,0.0
1,10.0,1.0,1.0,3.0
2,8.0,1.0,1.0,2.0
3,25.0,3.0,3.0,1.0
4,12.0,1.0,0.0,1.0


Feature and Target Separation

The target variable is separated from the predictor variables before splitting the dataset.

In [14]:
leakage_columns = [
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_amnt",
    "out_prncp",
    "out_prncp_inv"
]

risk_df = risk_df.drop(
    columns=leakage_columns,
    errors="ignore"
)

In [15]:
X = risk_df.drop(columns=["target"])

y = risk_df["target"]

print("Feature Matrix :", X.shape)
print("Target Vector :", y.shape)

Feature Matrix : (1345310, 99)
Target Vector : (1345310,)


Train Test Split

A stratified train-test split is performed to preserve the original class distribution in both datasets.

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [17]:
print("Training Set :", X_train.shape)
print("Testing Set :", X_test.shape)

print("\nTraining Target Distribution")
print(y_train.value_counts(normalize=True))

print("\nTesting Target Distribution")
print(y_test.value_counts(normalize=True))

Training Set : (1076248, 99)
Testing Set : (269062, 99)

Training Target Distribution
target
0    0.800374
1    0.199626
Name: proportion, dtype: float64

Testing Target Distribution
target
0    0.800373
1    0.199627
Name: proportion, dtype: float64


Nominal Feature Identification

Only nominal categorical variables require One-Hot Encoding. Ordinal and binary features have already been encoded.

In [18]:
nominal_features = [
    "purpose",
    "home_ownership",
    "verification_status",
    "addr_state"
]

print(nominal_features)

['purpose', 'home_ownership', 'verification_status', 'addr_state']


One Hot Encoding

The encoder is fitted only on the training dataset to prevent data leakage.

In [19]:
onehot_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

In [20]:
encoded_train = onehot_encoder.fit_transform(
    X_train[nominal_features]
)

encoded_test = onehot_encoder.transform(
    X_test[nominal_features]
)

In [21]:
encoded_feature_names = onehot_encoder.get_feature_names_out(
    nominal_features
)

encoded_train = pd.DataFrame(
    encoded_train,
    columns=encoded_feature_names,
    index=X_train.index
)

encoded_test = pd.DataFrame(
    encoded_test,
    columns=encoded_feature_names,
    index=X_test.index)

In [22]:
X_train = X_train.drop(
    columns=nominal_features
)

X_test = X_test.drop(
    columns=nominal_features
)

X_train = pd.concat(
    [X_train, encoded_train],
    axis=1
)

X_test = pd.concat(
    [X_test, encoded_test],
    axis=1
)

In [23]:
print(X_train.shape)

print(X_test.shape)

(1076248, 169)
(269062, 169)


Missing Value Validation

The training and testing datasets are inspected to ensure that no unexpected missing values remain after encoding.

In [24]:
missing_train = X_train.isnull().sum()

missing_train = missing_train[
    missing_train > 0
].sort_values(
    ascending=False
)

missing_train

mths_since_last_record            893587
mths_since_recent_bc_dlq          820882
mths_since_last_major_derog       793273
mths_since_recent_revol_delinq    716093
il_util                           704049
mths_since_rcnt_il                657512
all_util                          646217
inq_last_12m                      646175
total_cu_tl                       646175
open_acc_6m                       646175
total_bal_il                      646174
max_bal_bc                        646174
open_rv_24m                       646174
open_rv_12m                       646174
inq_fi                            646174
open_il_24m                       646174
open_il_12m                       646174
open_act_il                       646174
mths_since_last_delinq            542634
pct_tl_nvr_dlq                     54303
avg_cur_bal                        54192
num_rev_accts                      54176
mo_sin_rcnt_rev_tl_op              54175
num_actv_bc_tl                     54175
num_accts_ever_1

In [25]:
missing_test = X_test.isnull().sum()

missing_test = missing_test[
    missing_test > 0
].sort_values(
    ascending=False
)

missing_test

mths_since_last_record            223168
mths_since_recent_bc_dlq          205408
mths_since_last_major_derog       198287
mths_since_recent_revol_delinq    179255
il_util                           176245
mths_since_rcnt_il                164416
all_util                          161548
total_bal_il                      161538
inq_last_12m                      161538
total_cu_tl                       161538
inq_fi                            161538
open_rv_24m                       161538
open_rv_12m                       161538
max_bal_bc                        161538
open_il_24m                       161538
open_act_il                       161538
open_acc_6m                       161538
open_il_12m                       161538
mths_since_last_delinq            136109
pct_tl_nvr_dlq                     13378
avg_cur_bal                        13357
mo_sin_old_rev_tl_op               13353
mo_sin_rcnt_rev_tl_op              13353
num_tl_op_past_12m                 13352
num_tl_90g_dpd_2

Missing Value Treatment

Missing values are handled using a business-aware strategy.

- Features where missing values represent the absence of an event are filled with **-1**.
- Remaining numerical features are imputed using the median computed from the training data.

In [26]:
business_missing = [
    "mths_since_last_record",
    "mths_since_recent_bc_dlq",
    "mths_since_last_major_derog",
    "mths_since_recent_revol_delinq",
    "mths_since_last_delinq",
    "mths_since_recent_inq",
    "mths_since_rcnt_il",
    "mo_sin_old_il_acct"
]

In [27]:
for column in business_missing:

    if column in X_train.columns:

        X_train[column] = X_train[column].fillna(-1)

        X_test[column] = X_test[column].fillna(-1)

In [28]:
numeric_columns = X_train.select_dtypes(
    include=["int64", "float64"]
).columns

In [29]:
median_values = X_train[numeric_columns].median()

X_train[numeric_columns] = X_train[numeric_columns].fillna(
    median_values
)

X_test[numeric_columns] = X_test[numeric_columns].fillna(
    median_values
)

In [30]:
print(
    X_train.isnull().sum().sum()
)

print(
    X_test.isnull().sum().sum()
)

0
0


Credit History Feature

The `earliest_cr_line` feature represents the date when the borrower first established a credit account.

Instead of treating it as a categorical value, it is converted into the borrower's credit history length in years. This provides a more meaningful numerical representation of credit experience.

In [31]:
X_train["earliest_cr_line"] = pd.to_datetime(
    X_train["earliest_cr_line"],
    format="%b-%Y"
)

X_test["earliest_cr_line"] = pd.to_datetime(
    X_test["earliest_cr_line"],
    format="%b-%Y"
)

In [32]:
reference_date = pd.Timestamp("2018-12-01")

credit_history_train = (
    (reference_date - X_train["earliest_cr_line"]).dt.days / 365.25
)

credit_history_test = (
    (reference_date - X_test["earliest_cr_line"]).dt.days / 365.25
)

In [33]:
X_train = X_train.drop(
    columns=["earliest_cr_line"]
).assign(
    credit_history_years=credit_history_train
)

X_test = X_test.drop(
    columns=["earliest_cr_line"]
).assign(
    credit_history_years=credit_history_test
)

X_train = X_train.copy()

X_test = X_test.copy()

print("Credit history feature created successfully.")

/var/folders/0p/b4wn1lpd10l1n6_pslnp1csw0000gq/T/ipykernel_22793/3279953388.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ).assign(
/var/folders/0p/b4wn1lpd10l1n6_pslnp1csw0000gq/T/ipykernel_22793/3279953388.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ).assign(


Credit history feature created successfully.


In [34]:
print("Training Dataset")

print(
    X_train["credit_history_years"].describe()
)

print()

print("Testing Dataset")

print(
    X_test["credit_history_years"].describe()
)

Training Dataset
count    1.076248e+06
mean     1.974097e+01
std      7.604195e+00
min      3.249829e+00
25%      1.449966e+01
50%      1.833265e+01
75%      2.366872e+01
max      8.466804e+01
Name: credit_history_years, dtype: float64

Testing Dataset
count    269062.000000
mean         19.762448
std           7.631364
min           3.167693
25%          14.499658
50%          18.332649
75%          23.668720
max          74.915811
Name: credit_history_years, dtype: float64


Feature Scaling

Numerical variables are standardized using StandardScaler.

The scaler is fitted only on the training dataset and then applied to the testing dataset to prevent data leakage.

In [35]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Numeric Features :", len(numeric_features))

Numeric Features : 169


In [36]:
scaler = StandardScaler()

X_train[numeric_features] = scaler.fit_transform(
    X_train[numeric_features]
)

X_test[numeric_features] = scaler.transform(
    X_test[numeric_features]
)

print("Feature scaling completed.")

Feature scaling completed.


In [37]:
X_train[numeric_features].describe().T[
    ["mean", "std"]
].head(15)

,mean,std
loan_amnt,4.235866e-17,1.0
funded_amnt,-3.897842e-17,1.0
funded_amnt_inv,-4.605580e-17,1.0
int_rate,-7.938287e-17,1.0
installment,9.644253e-17,1.0
sub_grade,9.844955e-17,1.0
annual_inc,4.647833e-18,1.0
dti,-1.120762e-16,1.0
delinq_2yrs,-4.172486e-18,1.0
fico_range_low,2.968275e-16,1.0


In [38]:
X_train.to_parquet(
    "../data/processed/X_train.parquet",
    index=False
)

X_test.to_parquet(
    "../data/processed/X_test.parquet",
    index=False
)

y_train.to_frame().to_parquet(
    "../data/processed/y_train.parquet",
    index=False
)

y_test.to_frame().to_parquet(
    "../data/processed/y_test.parquet",
    index=False
)

print("=" * 60)
print("MODEL PREPARATION COMPLETED")
print("=" * 60)

print("Training Features :", X_train.shape)
print("Testing Features :", X_test.shape)
print("Training Labels :", y_train.shape)
print("Testing Labels :", y_test.shape)

MODEL PREPARATION COMPLETED
Training Features : (1076248, 169)
Testing Features : (269062, 169)
Training Labels : (1076248,)
Testing Labels : (269062,)


Model Preparation Summary

Completed Tasks

- Removed redundant and leakage-prone features.
- Applied binary encoding.
- Applied ordinal encoding.
- Applied one-hot encoding.
- Converted earliest credit line into credit history length.
- Applied business-aware missing value treatment.
- Standardized numerical features.
- Performed stratified train-test split.
- Generated final machine learning datasets.

Output Files

- X_train.parquet
- X_test.parquet
- y_train.parquet
- y_test.parquet

These datasets will be used directly during model training without repeating preprocessing.